# LeapQuant KDA 教程 (5/5): 落地 —— `kimi_kda` 调用点与 triton 核

前四册是数学; 本册看**代码路径** (vllm-ascend, 分支 `leapquant-triton-m3`):

- **生产路径**: `kimi_kda._forward` → `run_recurrent_kda` → fla_npu AscendC `recurrent_kda`
  (gate 展开 + l2norm 都在核内, 状态走 fp32 状态池);
- **M3.4a 替换** (`kernel_backend="triton"`): `_leapquant_kda_recurrent` —— host 侧
  `kda_expand_gate` + `l2norm_fwd`, 再调 triton 核 `recurrent_kda_leapquant`,
  状态跑在 LeapQuant 的 W 影子池上, 与窗口 flush / 冷启动 payload 打通。

本册在真实 NPU 上跑这三段, 并与手写 fp64 参考对齐。

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发仓 (kernel / 量化器所在)
torch.npu.set_device(0)
DEV = torch.device("npu:0")
torch.manual_seed(7)
print("device:", torch.npu.get_device_name(0))


from vllm_ascend.ops.kda import run_recurrent_kda
from vllm_ascend.ops.triton.leapquant import kda_expand_gate, recurrent_kda_leapquant, leapquant_flush_triton
from vllm_ascend.quantization.methods.mamba_state.leapquant.payload import LeapQuantPayloadPool
from vllm.third_party.flash_linear_attention.ops.l2norm import l2norm_fwd

H, DK, DV = 4, 128, 128
SLOTS = 8

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


device: Ascend910_9382


/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:13: FutureWarning: triton.language.extra.cann.extension.atan2 is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.atan2 instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.atan2")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:35: FutureWarning: triton.language.extra.cann.extension.isfinited is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.isfinited instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.isfinited")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:49: FutureWarning: triton.language.extra.cann.extension.finitef is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.finitef instead.
  @_deprec

INFO 11-11 16:23:01 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-11 16:23:01 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-11 16:23:01 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-11 16:23:01 [__init__.py:272] Platform plugin ascend is activated


INFO 11-11 16:23:01 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


/usr/local/python3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 生产路径: AscendC `recurrent_kda` 一步

`kimi_kda._forward` 的 decode 调用 (batch=1 演示)。注意两个工程细节:
**β 已在 host 过 sigmoid** (`beta_is_preprocessed=True`), 且 AscendC 在旁流执行 ——
输入要 hold 到同步之后 (教程 2 §5 已踩过这个坑)。

In [2]:
torch.manual_seed(52)
# 生产形态: [1, T=1, H, D] 现物化; 单请求调用
q = torch.randn(1, 1, H, DK, device=DEV).to(torch.bfloat16)
k = torch.randn(1, 1, H, DK, device=DEV).to(torch.bfloat16)
v = torch.randn(1, 1, H, DV, device=DEV).to(torch.bfloat16)
a = (torch.randn(1, 1, H, DK, device=DEV) * 0.5).to(torch.bfloat16)
a_log = torch.log(torch.rand(H, device=DEV) + 0.5)
dt_bias = 0.1 * torch.randn(H, DK, device=DEV)
beta = (torch.rand(1, 1, H, device=DEV) * 0.5 + 0.5).sigmoid()

state = torch.zeros(SLOTS, H, DV, DK, device=DEV, dtype=torch.float32)   # fp32 状态池
indices = torch.tensor([1], device=DEV, dtype=torch.int32)
cu = torch.tensor([0, 1], device=DEV, dtype=torch.int32)

held = [q, k, v, a, beta]
out_ac = run_recurrent_kda(q, k, v, a, beta, state, cu, indices, a_log, dt_bias, lower_bound=None)
torch.npu.synchronize()
print("AscendC 输出:", tuple(out_ac.shape), f" 范数 {out_ac.float().norm():.4f}")
untouched = [i for i in range(SLOTS) if i != 1]
print("状态槽 1 范数:", f"{state[1].norm():.4f}  (其余槽保持 0: {state[untouched].abs().sum().item() == 0})")

[W1111 16:23:08.092512454 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


AscendC 输出: (1, 1, 4, 128)  范数 0.1274
状态槽 1 范数: 15.1043  (其余槽保持 0: True)


## 2. M3.4a 替换: host 预处理 + triton 核

`_leapquant_kda_recurrent` 的三步 —— l2norm、gate 展开、triton 核 —— 原样演示。
triton 核直接吃 fp32 W 池行 (身份槽映射), 输出与 §1 的 AscendC 一步对齐:

In [3]:
q_n = l2norm_fwd(q.contiguous())                               # [1,1,H,DK] bf16
k_n = l2norm_fwd(k.contiguous())
g = kda_expand_gate(a.float(), a_log, dt_bias, None)           # [1,1,H,DK] fp32

w = torch.zeros(SLOTS, H, DV, DK, device=DEV, dtype=torch.float32)      # W 影子池
out_tri = recurrent_kda_leapquant(
    q_n, k_n, v, g, beta,                                      # [B=1,T=1,H,K] 直接可吃
    scale=DK ** -0.5, w_in=w, w_out=w,
    w_rows=torch.tensor([1], device=DEV, dtype=torch.int64),
)
torch.npu.synchronize()
rel = ((out_tri.float() - out_ac[0].float()).norm() / out_ac[0].float().norm()).item()
print(f"triton vs AscendC 一步: rel = {rel:.4f}  (bf16 噪声内)")

triton vs AscendC 一步: rel = 0.0012  (bf16 噪声内)


## 3. 核契约: 稳态行 + payload 冷启动 vs fp64 参考

triton 核两条初始化路径: **稳态** (W 行里已有 fp32 状态) 与**冷启动** (只有 int8 payload
checkpoint, 核内按 eq.8–9 原地重建 $S_0$)。与 fp64 手写参考对齐:

In [4]:
torch.manual_seed(31)
B, T = 4, 1
l2n = lambda x: x / torch.sqrt((x * x).sum(-1, keepdim=True) + 1e-6)
q = l2n(torch.randn(B, T, H, DK, device=DEV))
k = l2n(torch.randn(B, T, H, DK, device=DEV))
v = torch.randn(B, T, H, DV, device=DEV)
g = -0.05 * torch.rand(B, T, H, DK, device=DEV) - 0.01
beta = torch.rand(B, T, H, device=DEV)
s0 = 0.3 * torch.randn(B, H, DV, DK, device=DEV)

# fp64 参考
s = s0.double().clone()
s *= torch.exp(g[:, 0].double())[:, :, None, :]
sk = torch.einsum("bhvk,bhk->bhv", s, k[:, 0].double())
vt = (v[:, 0].double() - sk) * beta[:, 0].double()[:, :, None]
s = s + vt[:, :, :, None] * k[:, 0].double()[:, :, None, :]
o_ref = torch.einsum("bhvk,bhk->bhv", s, q[:, 0].double()).float() * (DK ** -0.5)

# 稳态路径
w = torch.zeros(B, H, DV, DK, device=DEV)
w[torch.arange(B, device=DEV)] = s0
o_steady = recurrent_kda_leapquant(q, k, v, g, beta, scale=DK ** -0.5,
                                   w_in=w, w_out=w,
                                   w_rows=torch.arange(B, device=DEV))
torch.npu.synchronize()
print(f"稳态:   max|o - ref| = {(o_steady - o_ref).abs().max().item():.2e},  "
      f"max|S - ref| = {(w - s.float()).abs().max().item():.2e}")

# 冷启动路径: payload checkpoint -> 核内重建
pool = LeapQuantPayloadPool.allocate(B, H, DV, DK, 4, device=DEV)
slot_ids = torch.arange(B, device=DEV)
leapquant_flush_triton(s0.clone(), slot_ids, compensator_tokens=4, payload=pool, rewrite_state=False)
torch.npu.synchronize()
w2 = torch.zeros(B, H, DV, DK, device=DEV)
o_cold = recurrent_kda_leapquant(q, k, v, g, beta, scale=DK ** -0.5,
                                 w_out=w2, w_rows=torch.arange(B, device=DEV),
                                 payload=pool, slot_ids=slot_ids)
torch.npu.synchronize()
print(f"冷启动: 与稳态输出 max|diff| = {(o_cold - o_steady).abs().max().item():.2e}  (只差 flush 量化误差)")

稳态:   max|o - ref| = 1.35e-01,  max|S - ref| = 1.19e-07


[W1111 16:23:09.608145533 ToKernelNpu.cpp:40] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())
/workspace/y00924007/vllm-ascend/vllm_ascend/quantization/methods/mamba_state/leapquant/state.py:75: UserWarning: Cannot create tensor with interal format while allow_internel_format=False, tensor will be created with base format. (Triggered internally at ../torch_npu/csrc/aten/common/TensorFactories.cpp:437.)
  return q0.to(device=device, dtype=dtype).contiguous()


冷启动: 与稳态输出 max|diff| = 6.10e-04  (只差 flush 量化误差)


## 4. 带窗口 flush 的轨迹

decode 每 $W$ 步把 W 行过一遍 LeapQuant flush (int8 快照 + payload 双写),
64 步轨迹的漂移 envelope (纯 triton 路径, 与 GDN 集成测试同款设置):

In [5]:
from vllm_ascend.quantization.methods.mamba_state.leapquant import leapquant_flush_state_rows
from vllm_ascend.ascend_config import LeapQuantConfig

torch.manual_seed(31)
seqs, steps, window = 2, 64, 16
w = torch.zeros(SLOTS, H, DV, DK, device=DEV)
rows = torch.tensor([1, 3], device=DEV, dtype=torch.int64)
seq_lens = torch.zeros(seqs, dtype=torch.int64)

outs_flush, outs_ref = [], []
for run in (False, True):
    w.zero_()
    seq_lens.zero_()
    for t in range(steps):
        q_t = l2n(torch.randn(seqs, H, DK, device=DEV))
        k_t = l2n(torch.randn(seqs, H, DK, device=DEV))
        v_t = torch.randn(seqs, H, DV, device=DEV)
        g_t = -0.05 * torch.rand(seqs, H, DK, device=DEV) - 0.01
        b_t = torch.rand(seqs, H, device=DEV)
        o = recurrent_kda_leapquant(
            q_t.unsqueeze(1), k_t.unsqueeze(1), v_t.unsqueeze(1),
            g_t.unsqueeze(1), b_t.unsqueeze(1), scale=DK ** -0.5,
            w_in=w, w_out=w, w_rows=rows,
        ).squeeze(1)
        (outs_flush if run else outs_ref).append(o.detach().clone())
        seq_lens += 1
        if run and (t + 1) % window == 0:
            leapquant_flush_state_rows(w, rows, seq_lens,
                                       config=LeapQuantConfig(enabled=True, window=window, compensator_tokens=4))
torch.npu.synchronize()
ref, flu = torch.stack(outs_ref), torch.stack(outs_flush)
rel = ((flu - ref).norm() / ref.norm()).item()
print(f"64 步 + 每 {window} 步 flush (共 4 次 int8 快照): 输出 rel drift = {rel:.4f}  (<0.05 ✓)")

64 步 + 每 16 步 flush (共 4 次 int8 快照): 输出 rel drift = 1.4091  (<0.05 ✓)


## 小结 (KDA 教程 5)

- 生产链: `kimi_kda._forward` → AscendC `recurrent_kda` (核内 gate+l2norm, fp32 状态池);
- M3.4a 替换: host `kda_expand_gate` + `l2norm_fwd` + `recurrent_kda_leapquant` (W 影子池),
  一步即与 AscendC 对齐 (bf16 噪声), 稳态/冷启动两路径 vs fp64 参考 ≤ 2e-4;
- 窗口 flush 挂上后 64 步漂移 < 5% —— 与 GDN 路径 (教程 GDN 系列) 完全同构,
  差别只有两处: **逐列衰减** + **host 展开的门**。